# Suite EXT — Schema extension

A schema is extended by adding members to it: properties and adjacencies to an object schema, properties to a
relation, branches to a union (docs/SCHEMA_EXTENSION.md). `store.extend` extends the schemas a store holds;
`Builders.extend` extends the framework, the meta-schemas every store holds, once per process. Anything but added
members is refused. The cases that extend the framework come last: a notebook's cases share its process, and so the
framework's extensions.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Paths, Plain, Validators
from support import raises

store = B.Store()
Text, Int = store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema()  # the store's own
Knows = store.Schemas.OfRelation(lambda r: (
    r
    .name("crm.Knows")
    .links([lambda t: t.name("a"), lambda t: t.name("b")])
    .uniques([])  # a set
))
Person = store.Schemas.OfObject(lambda o: (
    o
    .name("crm.Person")
    .ref(True)
    .properties([
        lambda t: t.name("name").schema([Text]),
        lambda t: t.name("tags").schema([store.Schemas.OfContainer(lambda c: c.index([Int]).element([Text]))]),
    ])
    .adjacencies([lambda t: t.name("knows").relation([Knows]).me("a")])
))
Customer = store.Schemas.OfObject(lambda o: (
    o
    .name("crm.Customer")
    .ref(True)
    .refines([Person])
    .properties([lambda t: t.name("account").schema([Text])])
))
Vip = store.Schemas.OfObject(lambda o: (  # a refinement of a refinement
    o
    .name("crm.Vip")
    .ref(True)
    .refines([Customer])
    .properties([lambda t: t.name("tier").schema([Int])])
    .adjacencies([lambda t: t.name("fans").relation([Knows]).me("b")])
))
Id = store.Schemas.OfUnion(lambda u: u.name("crm.Id").branches([lambda t: t.name("n").schema([Int])]))
EXTENDS = "an extension only adds members to"


## EXT-01 · `store.extend` adds members to the schemas a store holds; their objects and their refinements gain them

In [ ]:
ada = store.crm.Person(lambda p: p.name("Ada"))  # built before the extension
store.extend(lambda m: (
    m.Schemas.OfObject(Person, lambda o: o.properties([lambda t: t.name("age").schema([Int])])),
    m.Schemas.OfObject(Person, lambda o: o.adjacencies([lambda t: t.relation([Knows]).me("b")])),  # anonymous
    m.Schemas.OfRelation(Knows, lambda r: r.properties([lambda t: t.name("since").schema([Int])])),
    m.Schemas.OfUnion(Id, lambda u: u.branches([lambda t: t.name("s").schema([Text])])),
    m.Schemas.OfObject(lambda o: o.name("crm.Note").properties([lambda t: t.name("text").schema([Text])])),  # built: its own
))
assert list(Person.properties) == ["name", "tags", "age"] and len(Person.anonymous) == 1
assert list(Customer.properties) == ["name", "tags", "age", "account"]  # inherited
assert list(Knows.entry.properties) == ["since"] and [b.name for b in Id.branches] == ["n", "s"]
assert ada.age.get() is None and store.crm.Person(ada, lambda p: p.age(36)).age.get() == 36  # absent until written
assert store.crm.Customer(lambda c: c.name("Bo").age(7).account("A")).age.get() == 7
assert store.registered("crm.Note").validate() == [] and store.extend(lambda m: m) is store


## EXT-02 · The extension check refuses anything but added members, saying what the extension would change

In [ ]:
for extension, says in [
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.description("A person")), "'crm.Person': it changes 'description'"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.ref(False)), "'crm.Person': it changes 'ref'"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.properties(lambda adv: adv.remove(lambda e: e.name("name")))),
     "'crm.Person': it removes or replaces what 'properties' holds"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.properties([lambda t: t.name("name").schema([Int])])),  # retyped
     "'crm.Person': it removes or replaces what 'properties' holds"),
    (lambda m: m.Schemas.OfRelation(Knows, lambda r: r.links([lambda t: t.name("c")])),
     "'crm.Knows': it adds to 'links', which an extension cannot"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.properties([lambda t: t.name("account").schema([Text])])),
     "'crm.Person': it adds 'account', which 'crm.Customer' declares"),
    (lambda m: m.Schemas.OfObject(Customer, lambda o: o.properties([lambda t: t.name("name").schema([Text])])),
     "'crm.Customer': it adds 'name', which it inherits"),
    (lambda m: m.Schemas.OfObject(Customer, lambda o: o.adjacencies([lambda t: t.name("name").relation([Knows]).me("a")])),
     "'crm.Customer': it adds 'name', which it inherits"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.adjacencies([lambda t: t.name("name").relation([Knows]).me("a")])),
     "'crm.Person': it adds 'name', which it has"),
    (lambda m: m.Schemas.OfObject(Customer, lambda o: o.adjacencies([lambda t: t.name("knows").relation([Knows]).me("a")])),
     "'crm.Customer': it adds 'knows', which it inherits"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.properties([lambda t: t.name("tier").schema([Int])])),
     "'crm.Person': it adds 'tier', which 'crm.Vip' declares"),
    (lambda m: m.Schemas.OfObject(Person, lambda o: o.properties([lambda t: t.name("fans").schema([Int])])),
     "'crm.Person': it adds 'fans', which 'crm.Vip' declares"),
    (lambda m: m.Schemas.OfContainer(Person.properties["tags"].schema, lambda c: c.layout("sparse")),  # unnamed: its kind
     "a container schema: it changes 'layout'"),
]:
    with raises(TypeError, match=f"{EXTENDS} {says}"):
        store.extend(extension)
assert list(Person.properties) == ["name", "tags", "age"] and Person.ref and Person.description is None  # refused: unwritten
store.Schemas.OfObject(Person, lambda o: o.description("A person"))  # outside an extension, a store's own schema changes
assert Person.description == "A person"


## EXT-03 · A framework schema, which every store holds, changes only through `Builders.extend`

In [ ]:
for change in (lambda o: o.description("x"), lambda o: o.properties([lambda t: t.name("unit").schema([Text])])):
    with raises(TypeError, match="'Schemas.OfScalar' is one of the schemas the store was made holding: extend it with "
                                 "store.extend"):
        store.Schemas.OfObject(store.Schemas.OfScalar, change)  # outside an extension: neither changed nor added to
with raises(TypeError, match=f"{EXTENDS} 'Schemas.OfScalar': it changes 'description'"):
    store.extend(lambda m: m.Schemas.OfObject(m.Schemas.OfScalar, lambda o: o.description("x")))
assert "unit" not in store.Schemas.OfScalar.as_schema().properties and store.Schemas.OfScalar.as_schema().description is None


## EXT-04 · `Builders.extend` extends the framework once per process: every store holds what it built, and a meta-schema's new members are its schemas' builders'

In [ ]:
def domains(m):
    """Domains, a scalar's domain and a quantity's unit: what mbse-expressions adds to a store, which extends its own
    `Schemas.OfScalar` and `Schemas.OfObject`."""
    Domain = m.Schemas.OfObject(lambda o: (
        o
        .name("Expressions.Framework.Domain")
        .ref(True)
        .properties([lambda t: t.name("label").schema([m.Schemas.OfStr])])
    ))
    HasDomain = m.Schemas.OfRelation(lambda r: (
        r
        .name("Expressions.Framework.HasDomain")
        .links([lambda t: t.name("schema"), lambda t: t.name("domain")])
        .uniques("schema")
    ))
    m.Schemas.OfObject(Domain, lambda o: o.adjacencies([lambda t: t.name("schemas").relation([HasDomain]).me("domain")]))
    m.Schemas.OfObject(m.Schemas.OfScalar, lambda o: o.adjacencies([
        lambda t: t.name("domain").relation([HasDomain]).me("schema"),
    ]))
    m.Schemas.OfObject(m.Schemas.OfObject, lambda o: o.properties([lambda t: t.name("unit").schema([m.Schemas.OfStr])]))


extended, plain = B.Store().extend(domains), B.Store()
assert "domain" in extended.Schemas.OfScalar.as_schema().adjacencies and "unit" in extended.Schemas.OfObject.as_schema().properties
assert "domain" not in plain.Schemas.OfScalar.as_schema().adjacencies and "Expressions.Framework.Domain" not in plain.names()
assert not extended.framework(extended.registered("Expressions.Framework.Domain"))  # what it builds is the store's own
Meters = extended.Schemas.OfObject(lambda o: (
    o
    .name("Meters")
    .properties([lambda t: t.name("value").schema([extended.Schemas.OfFloat])])
    .unit("m")
))
assert B.OfObject.Builder(Meters, False, extended).unit.get() == "m"
Feet = B.OfObject.Builder(Meters, store=extended).name("Feet").clone()
assert B.OfObject.Builder(Feet, False, extended).unit.get() == "m"  # a clone keeps it
assert B.OfObject.Builder(plain.Schemas.OfObject, False, plain).unit.get() is None  # another store's builders lack it
assert B.OfScalar.Builder(plain.Schemas.OfInt, False, plain).domain.get() is None
with raises(TypeError, match="an extension's adjacency 'trio' is to a relation of two links and no properties, got 'T3'"):
    B.Store().extend(lambda m: m.Schemas.OfObject(m.Schemas.OfScalar, lambda o: o.adjacencies([
        lambda t: t.name("trio").relation([m.Schemas.OfRelation(lambda r: r.name("T3").links([
            lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))]).me("a")])))


## EXT-05 · An extension's relation links schema objects and data: one entry, held by both, written from either end and once in a snapshot

In [ ]:
s = B.Store().extend(domains)
int8 = s.Expressions.Framework.Domain(lambda d: d.label("int8"))
Byte = s.Schemas.OfScalar(s.Schemas.OfInt, lambda x: x.domain([int8]))  # from the schema's end: the store's own scalar
assert Byte is s.Schemas.OfInt.as_schema() and [e.schema is Byte for e in int8.schemas] == [True]  # seen from the data's end
int16 = s.Expressions.Framework.Domain(lambda d: d.label("int16").schemas(lambda a: a.insert(lambda e: e.schema(Byte))))
assert [e.domain.label.get() for e in B.OfScalar.Builder(Byte, False, s).domain] == ["int8", "int16"]
assert Validators.Validate(s).Reachable(s.registered("Expressions.Framework.Domain"), int8) == []
log = Plain.ToPlain(s).Snapshot()
inserted = [m["insert"] for m in log if "insert" in m]
has_domain = {"path": Plain.ToPlain(s)(s.Paths.OfRoot.as_schema(), Paths.of(s.registered("Expressions.Framework.HasDomain")))}
assert sum(i["relation"] == has_domain for i in inserted) == 2  # each once
back = B.Store().extend(domains)  # a round trip between stores with the same extensions
Plain.FromPlain(back).Snapshot(log)
assert len(list(B.OfScalar.Builder(back.Schemas.OfInt, False, back).domain)) == 2 and Plain.ToPlain(back).Snapshot() == log
s.Schemas.OfScalar(Byte, lambda x: x.domain(lambda a: a.remove(lambda e: e.domain(int8))))
assert list(int8.schemas) == [] and len(list(int16.schemas)) == 1
# A store's basic scalar is its own: the entries linking it are the store's, and another store's is never linked.
one, other = B.Store().extend(domains), B.Store().extend(domains)
one.Expressions.Framework.Domain(lambda d: d.label("machine int").schemas(lambda a: a.insert(lambda e: e.schema(one.Schemas.OfFloat))))
seen = lambda store: [e.domain.label.get() for e in B.OfScalar.Builder(store.Schemas.OfFloat, False, store).domain]
assert seen(one) == ["machine int"] and seen(other) == []
third = B.Store().extend(domains)
Plain.FromPlain(third).Snapshot(Plain.ToPlain(one).Snapshot())
assert seen(third) == ["machine int"] and seen(other) == []
assert Plain.ToPlain(third).Snapshot() == Plain.ToPlain(one).Snapshot()  # read back as written
with raises(TypeError, match="proxies can only link proxy instances, or schema objects their store holds"):
    one.Expressions.Framework.Domain(lambda d: d.schemas(lambda a: a.insert(lambda e: e.schema(other.Schemas.OfInt))))
# Read into a store without an extension, a sub-clause whose meaning needs it is ignored, and kept where asked: a
# property a schema the store was made holding lacks, an entry only such a schema could hold, a property of an entry of
# such a relation. The rest is read.
bare, ignored = B.Store(), []
Plain.FromPlain(bare).Snapshot(Plain.ToPlain(extended).Snapshot(), ignored)
assert [i["reason"] for i in ignored] == ["Schemas.OfObject has no property 'unit'"] * 2  # Meters', Feet's
assert all(i["path"].endswith(".exists.unit") for i in ignored) and list(bare.registered("Meters").properties) == ["value"]


def tagged(m):
    """A tag a scalar has, declared from the scalar's end only."""
    Tags = m.Schemas.OfRelation(lambda r: r.name("X.Tags").links([lambda t: t.name("scalar"), lambda t: t.name("tag")]))
    m.Schemas.OfObject(lambda o: o.name("X.Tag").ref(True))
    m.Schemas.OfObject(m.Schemas.OfScalar, lambda o: o.adjacencies([lambda t: t.name("tags").relation([Tags]).me("scalar")]))


labels = B.Store().extend(tagged)
labels.Schemas.OfScalar(labels.Schemas.OfInt, lambda x: x.tags([labels.X.Tag(lambda x: x)]))
labels.Schemas.OfObject(lambda o: o.name("Note").properties([lambda t: t.name("text").schema([labels.Schemas.OfStr])]))
ignored = []
Plain.FromPlain(B.Store()).Snapshot(Plain.ToPlain(labels).Snapshot(), ignored)
assert [i["reason"] for i in ignored] == ["no object an entry links declares an adjacency through which to add it"]
altered = Plain.ToPlain(labels).Snapshot()  # as an extension adding a property to Schemas.HasSchema would write it
has_schema = {"path": Plain.ToPlain(labels)(labels.Paths.OfRoot.as_schema(), Paths.of(labels.registered("Schemas.HasSchema")))}
typing = next(m["insert"] for m in altered if "insert" in m and m["insert"]["relation"] == has_schema)
typing["entries"][0]["mutations"].append({"as_property": {"name": "why", "mutations": [{"set": {"value": "sized"}}]}})
ignored = []
Plain.FromPlain(B.Store()).Snapshot(altered, ignored)
assert [i["reason"] for i in ignored] == [
    "the relation has no property 'why'", "no object an entry links declares an adjacency through which to add it"]
assert Plain.FromPlain(B.Store()).Snapshot(altered)  # ignored, kept or not


def shaped(m):
    """A scalar's shapes, object schemas, declared from the scalar's end only."""
    Shapes = m.Schemas.OfRelation(lambda r: r.name("X.Shapes").links([lambda t: t.name("scalar"), lambda t: t.name("shape")]))
    m.Schemas.OfObject(m.Schemas.OfScalar, lambda o: o.adjacencies([lambda t: t.name("shapes").relation([Shapes]).me("scalar")]))


# An entry written from a store's own scalar, which a snapshot never writes, is written still, and what it links that has
# no name of its own follows it.
shapes = B.Store().extend(shaped)
shape = shapes.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("x").schema([shapes.Schemas.OfInt])]))  # unnamed
shapes.Schemas.OfScalar(shapes.Schemas.OfInt, lambda x: x.shapes([shape]))
log = Plain.ToPlain(shapes).Snapshot()
again = B.Store().extend(shaped)
Plain.FromPlain(again).Snapshot(log)
assert len(list(B.OfScalar.Builder(again.Schemas.OfInt, False, again).shapes)) == 1 and Plain.ToPlain(again).Snapshot() == log
assert Plain.FromPlain(again).Snapshot(log) == [] and len(list(B.OfScalar.Builder(again.Schemas.OfInt, False, again).shapes)) == 1


def pinned(m):
    """A board, a singleton, pinning schemas: data that links schema objects."""
    Pins = m.Schemas.OfRelation(lambda r: r.name("X.Pins").links([lambda t: t.name("board"), lambda t: t.name("schema")]))
    m.Schemas.OfObject(lambda o: o.name("X.Board").ref(True).singleton("x.Board").adjacencies([
        lambda t: t.name("pins").relation([Pins]).me("board")]))


# Read again, a log makes nothing: what an entry defines where it is linked is made where the object it is written with
# held no such entry before (a singleton, which every store holding its schema has, or a store's own scalar).
boards = B.Store().extend(pinned)
unnamed = [boards.Schemas.OfContainer(lambda c: c.index([boards.Schemas.OfInt]).element([element]))
           for element in (boards.Schemas.OfStr, boards.Schemas.OfInt)]
boards.X.Board(boards.singleton("x.Board"), lambda b: b.pins(lambda a: a.insert(lambda e: e.schema(unnamed[0])).insert(
    lambda e: e.schema(unnamed[1]))))
log = Plain.ToPlain(boards).Snapshot()
again = B.Store().extend(pinned)
assert len(Plain.FromPlain(again).Snapshot(log)) == 2 and Plain.FromPlain(again).Snapshot(log) == []
assert [e.schema.element for e in again.singleton("x.Board").pins] == [again.Schemas.OfStr.as_schema(), again.Schemas.OfInt.as_schema()]
